# Inference Lab: Pricing Under Uncertainty


**Run this notebook in Google Colab.** The `demand` module is compiled for Linux and Python 3.13 only, so it will not import on a Mac or Windows machine.

The goal of this lab is to use what you've learned in this class to find the optimal price for something. You can't know for sure what the underlying curve of revenue vs price looks like. All you know is what you would know in the real world, information you gain by sampling-- you can market test any price using the demand function below. When you set the price, the market test runs for a certain amount of time, and then reports responses-- a list of 1s (bought) and 0s (didn't buy) for that price point. The total number in the sample, $n$, is therefore a random variable, given by `len(responses)`. The number of consumers who buy the product is `responses.sum().` Every time you run a market test, it takes time (literally, it will take a while to run, so don't do this lab last minute!).

In [4]:
!git clone -q https://github.com/smgroves/demand_lab.git

%cd /content/demand_lab

from demand import demand
import numpy as np

responses = demand(price=14.0)

print(len(responses), responses.sum())

/content/demand_lab
184 143


## 1. Compare two prices

Run one market test with price = 30. Then test price = 25. Which price earns more revenue per potential customer?

1. For each price, estimate the proportion $\hat{p}$ of customers who buy, and the revenue per customer $\hat{r}$.
2. Estimate the standard error of each $\hat{r}$ by bootstrapping.
3. Compute a 95% confidence interval for each $\hat{r}$.
4. Compute a 95% confidence interval for the difference $\hat{r}_{25} - \hat{r}_{30}$.
   
5. Interpret the estimates, SEs, and intervals. Report which price earns more profit, and how confident you are.



In [5]:
def revenue_per_customer(responses):
  cust = responses.sum()
  proportion = cust/len(responses)
  revenue = price*proportion

  return proportion, revenue

In [6]:
price = 30
responses_30 = demand(price)

props_30 = []
revs_30 = []
for i in range(5000):
  samp = np.random.choice(responses_30, size=len(responses_30), replace=True)
  p, r = revenue_per_customer(samp)

  props_30.append(p)
  revs_30.append(r)

props_30 = np.array(props_30)
revs_30 = np.array(revs_30)

In [7]:
price = 25
responses_25 = demand(price)

props_25 = []
revs_25 = []
for i in range(5000):
  samp = np.random.choice(responses_25, size=len(responses_25), replace=True)
  p, r = revenue_per_customer(samp)

  props_25.append(p)
  revs_25.append(r)

props_25 = np.array(props_25)
revs_25 = np.array(revs_25)

In [8]:
p_est_30 = props_30.mean()
r_est_30 = revs_30.mean()

p_est_25 = props_25.mean()
r_est_25 = revs_25.mean()

print("P=30:")
print(f"   p_hat = {p_est_30:.3f}")
print(f"   r_hat = ${r_est_30:.2f}")
print("\nP=25:")
print(f"   p_hat = {p_est_25:.3f}")
print(f"   r_hat = ${r_est_25:.2f}")

P=30:
   p_hat = 0.371
   r_hat = $11.14

P=25:
   p_hat = 0.419
   r_hat = $10.46


In [9]:
r_se_30 = revs_30.std()
r_se_25 = revs_25.std()

print(f"P=30: SE(r_hat) = {r_se_30:.3f}")
print(f"P=25: SE(r_hat) = {r_se_25:.3f}")

P=30: SE(r_hat) = 1.054
P=25: SE(r_hat) = 0.831


In [10]:
r_ci_30 = (r_est_30 - 1.96*r_se_30, r_est_30 + 1.96*r_se_30)
r_ci_25 = (r_est_25 - 1.96*r_se_25, r_est_25 + 1.96*r_se_25)

print(f"P=30: CI = ({r_ci_30[0]:.3f}, {r_ci_30[1]:.3f})")
print(f"P=25: CI = ({r_ci_25[0]:.3f}, {r_ci_25[1]:.3f})")

P=30: CI = (9.074, 13.207)
P=25: CI = (8.834, 12.093)


In [11]:
revs_diff = revs_25 - revs_30

r_est_diff = revs_diff.mean()
r_se_diff = revs_diff.std()

r_ci_diff = (r_est_diff - 1.96*r_se_diff, r_est_diff + 1.96*r_se_diff)

print(f"r_25 - r_30: SE = {r_se_diff:.3f}")
print(f"r_25 - r_30: CI = ({r_ci_diff[0]:.3f}, {r_ci_diff[1]:.3f})")

r_25 - r_30: SE = 1.348
r_25 - r_30: CI = (-3.318, 1.965)


The estimate for `price=$30` is \$11.14, which is slightly higher than the estimate for `price=$25` of \$10.46. However, the standard error at `price=$30` is higher than it is for `price=$25`, so it is possible \$25 is better. Our confidence interval for their difference includes 0, so we can't say with certainty which price is better.

## 2. The cost of sampling

7. Run a second market test at each price. Pool the two price = 30 samples (`np.concatenate`), and pool the two price = 25 samples.
8. Recompute the SEs, the intervals, and the interval for the difference. By roughly what factor did the SE shrink? Explain why, using the SE formula.
   
9.  **Was the extra sampling worth it?** Every market test takes time, and while you're testing you aren't selling at your best price, so each test costs the company revenue. Did the first round already settle the question (its difference interval excluded 0)? If so, the second round cost time and bought nothing but a narrower interval. If not, did the second round settle it?


In [12]:
price = 30
responses_30 = np.concatenate([responses_30, demand(price)])

props_30 = []
revs_30 = []
for i in range(5000):
  samp = np.random.choice(responses_30, size=len(responses_30), replace=True)
  p, r = revenue_per_customer(samp)

  props_30.append(p)
  revs_30.append(r)

props_30 = np.array(props_30)
revs_30 = np.array(revs_30)

In [13]:
price = 25
responses_25 = np.concatenate([responses_25, demand(price)])

props_25 = []
revs_25 = []
for i in range(5000):
  samp = np.random.choice(responses_25, size=len(responses_25), replace=True)
  p, r = revenue_per_customer(samp)

  props_25.append(p)
  revs_25.append(r)

props_25 = np.array(props_25)
revs_25 = np.array(revs_25)

In [14]:
p_est_30 = props_30.mean()
r_est_30 = revs_30.mean()

p_est_25 = props_25.mean()
r_est_25 = revs_25.mean()

print("P=30:")
print(f"   p_hat = {p_est_30:.3f}")
print(f"   r_hat = ${r_est_30:.2f}")
print("\nP=25:")
print(f"   p_hat = {p_est_25:.3f}")
print(f"   r_hat = ${r_est_25:.2f}")

P=30:
   p_hat = 0.364
   r_hat = $10.93

P=25:
   p_hat = 0.416
   r_hat = $10.40


In [15]:
r_se_30 = revs_30.std()
r_se_25 = revs_25.std()

print(f"P=30: SE(r_hat) = {r_se_30:.3f}")
print(f"P=25: SE(r_hat) = {r_se_25:.3f}")

P=30: SE(r_hat) = 0.750
P=25: SE(r_hat) = 0.632


In [16]:
r_ci_30 = (r_est_30 - 1.96*r_se_30, r_est_30 + 1.96*r_se_30)
r_ci_25 = (r_est_25 - 1.96*r_se_25, r_est_25 + 1.96*r_se_25)

print(f"P=30: CI = ({r_ci_30[0]:.3f}, {r_ci_30[1]:.3f})")
print(f"P=25: CI = ({r_ci_25[0]:.3f}, {r_ci_25[1]:.3f})")

P=30: CI = (9.462, 12.404)
P=25: CI = (9.164, 11.640)


In [17]:
revs_diff = revs_25 - revs_30

r_est_diff = revs_diff.mean()
r_se_diff = revs_diff.std()

r_ci_diff = (r_est_diff - 1.96*r_se_diff, r_est_diff + 1.96*r_se_diff)

print(f"r_25 - r_30: SE = {r_se_diff:.3f}")
print(f"r_25 - r_30: CI = ({r_ci_diff[0]:.3f}, {r_ci_diff[1]:.3f})")

r_25 - r_30: SE = 0.980
r_25 - r_30: CI = (-2.452, 1.390)


The original SE was 1.348, and the new SE was 0.980. If we divide 0.980/1.348, we get 0.727. Based on the formula for SE, which is $SE = \frac{σ}{\sqrt n}$, doubling n should multiply the SE by a factor of $\frac{1}{\sqrt 2}$. The square root of 2 is about 0.707, which is fairly close to the factor we observed.

Ultimately, the extra sampling was not worth it. Our new confidence interval, while about 30% narrower, still contains 0. Therefore, we can still not say with any statistical significance which price is better.

## 3. Finding the optimal price

Now find the price that maximizes revenue per customer. Search prices between about \$5 and \$100.

Every market test takes time, and time spent testing is revenue lost: while you test, you aren't selling at your best price. Your confidence intervals tell you whether another test is worth running. **Use as few market tests as you can (<25 would be good).** Your grade will be based on, in this order, whether you:
- accurately used the information you have by accurately interpreting confidence intervals
- found a good price, close to the optimal price
- and used fewer tests.

If this sounds fun to you, run with it! If you are still very confused about what to do, read the hints below.

<details><Summary>Hints</summary>
<b>Hint 1:</b> There are a few ways you could search:

- Brute force: test every price on a fine grid. This takes a market test at every price, including prices that are obviously bad. This would be a bad choice.
- Hill climbing: compare the CI for two prices, move toward the better one, and repeat. For example, start from the 25/30 comparison above; based on which price is better, choose another price \$10 away in that direction (15 if 25 was better; 40 if 30 was better). Shrink the step size as you get close to a good price. If you can't detect a difference between two prices, either shrink the step size or increase the sample size (i.e. run another market test at the same price).
- other strategies that I won't mention here.
  
<b>Hint 2:</b> You may want to try starting your tests with a few different prices across the range; there may be local maxima you will get stuck in if you just hill climb towards a single maximum.

<b>Hint 3:</b> You can make up your own strategy or try a mix of strategies. However, keep in mind that your grade will reflect how well you ran A/B testing and interpreting confidence intervals, more than how complex your strategy is. Similarly, a lucky guess with a brute force strategy would not be a good reproducible strategy.

<b>Hint 4:</b>
Turn your code from part 1 into a function, so you can easily input responses, run bootstrapping, calculate SE, and calculate confidence intervals. Then make another function where you give it two confidence intervals and make a decision based on them: sample more at the same test, or move in one direction with some step size.
   </details>

**Before you run any tests,** write down your plan: which prices you'll test first, how you'll decide where to go next, and what your intervals need to show before you stop. Then follow it, and note anywhere you changed course and why.

For any method, stop when your intervals show that another test is unlikely to change your answer by enough to matter.

**Deliverable:** A ~2-3 paragraph write up of the strategy you chose, your final recommended price, a 95% CI for $r$ at that price, the number of market tests you ran, and one or two sentences on how your intervals told you when to stop.

In [33]:
def revenue_mean(price):
  r = demand(price)
  revs = []

  for i in range(5000):
    samp = np.random.choice(r, size=len(r), replace=True)
    _, rev = revenue_per_customer(samp)

    revs.append(rev)

  revs = np.array(revs)

  M = revs.mean()
  return M, revs

def conf_interval(r1, r2):
  r_diff = r1-r2

  M = r_diff.mean()
  SE = r_diff.std()

  CI = (M-1.96*SE, M+1.96*SE)
  return CI

In [43]:
step_size = 25

max_index = 52.5
max_revenue, max_responses = revenue_mean(price=max_index)

interval_has_zero = False

while (not interval_has_zero) or (abs(step_size) > 0.1):
  m, r = revenue_mean(max_index+step_size)
  print(f"Price = {(max_index+step_size):.2f}, Revenue = ${m:.2f}")

  CI = conf_interval(r, max_responses)
  if CI[0] < 0 < CI[1]:
    interval_has_zero = True

  if m > max_revenue:
    max_index += step_size
    max_revenue = m
    max_responses = r

    step_size = (step_size/2)

  else:
    step_size = (-step_size/2)


Price = 77.50, Revenue = $7.30
Price = 40.00, Revenue = $9.13
Price = 33.75, Revenue = $10.46
Price = 30.62, Revenue = $9.72
Price = 35.31, Revenue = $10.31
Price = 32.97, Revenue = $10.45
Price = 34.14, Revenue = $9.41
Price = 33.55, Revenue = $9.13


In [49]:
print(f"best_price = ${max_index:.2f}, max_revenue = ${max_revenue:.2f}")

best_price = $33.75, max_revenue = $10.46


In [48]:
ci_low, ci_high = conf_interval(max_responses, 0)
print(f"({ci_low:.2f}, {ci_high:.2f})")

(8.73, 12.19)


My strategy was to use a simple algorithm that keeps track of the best value
and compares it with nearby values. For this, I used a step size which halves after each iteration. If the step is an improvement, it updates the maximum value, and moves in the same direction again. If the step is not an improvement, we stay at our current maximum and try a step in the opposite direction. For the confidence intervals, I stopped when 0 is present in the interval, and the step size is sufficiently small. I added the step size constraint since it was often finding 0 in the first iteration, given the high standard error. This is far from a perfect model, but I think it gives a quick way to find something close to the optimal value.

In total, this performed 9 market tests before stopping. After running the algorithm, the best price it found was \$33.75, which yielded a revenue per customer of \$10.46. Notably, this is actually lower than the revenue obtained at \$30 in part 1. This is due to the variability in each market test, which can greatly change from sample to sample. This algorithm is bootstrapping on only one sample, so it could vary dramatically based on the randomness of demand. In this run, the market test around \$30 yielded under \$10, so we landed at \$33.75. We can see the variability in the sampling mean through the confidence interval at \$33.75. The 95% confidence interval that was found is (8.73, 12.19). Therefore, there is a range of 3-4 dollars the revenue could reasonably land in. Overall, this model suffers from the high randomness of the market tests, which would ideally be run many times at each price. However, that was sacrificed in order to maintain speed and efficiency.